# Lab B — Watch it choose

Companion to *Latent Space*, Chapter 4.

**Makes visible:** there is no decision inside a language model, only a
probability distribution and a sampling procedure.

Runtime → Change runtime type → GPU makes this faster but it works on CPU.


In [ ]:
!pip -q install transformers accelerate

In [ ]:
# Writes this lab's files into Colab. Nothing to download, nothing
# to clone, nothing to upload. Just run this cell.
import base64, pathlib

FILES = {
    "watch_it_choose.py": (
        "IiIiCkxhYiBCOiB3YXRjaCB0aGUgbW9kZWwgY2hvb3NlLgoKVGhlcmUgaXMgbm8gZGVjaXNpb24gaW5zaWRlIGEgbGFuZ3VhZ2UgbW9kZWwuIFRoZXJlIGlz"
        "IGEgcHJvYmFiaWxpdHkKZGlzdHJpYnV0aW9uIG92ZXIgZXZlcnkgdG9rZW4gaXQga25vd3MsIGFuZCBhIHNhbXBsaW5nIHByb2NlZHVyZSB0aGF0IGRyYXdz"
        "CmZyb20gaXQuIFRoaXMgc2NyaXB0IGV4cG9zZXMgYm90aC4KClJ1bjogIHB5dGhvbiB3YXRjaF9pdF9jaG9vc2UucHkKICAgICAgcHl0aG9uIHdhdGNoX2l0"
        "X2Nob29zZS5weSAtLXByb21wdCAiVGhlIGNhcGl0YWwgb2YgRnJhbmNlIGlzIgogICAgICBweXRob24gd2F0Y2hfaXRfY2hvb3NlLnB5IC0tdGVtcGVyYXR1"
        "cmUgMS40CiAgICAgIHB5dGhvbiB3YXRjaF9pdF9jaG9vc2UucHkgLS1uZWFyLXRpZXMKIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN5cwoKUFJPTVBU"
        "UyA9IFsKICAgICJUaGUgY2FwaXRhbCBvZiBGcmFuY2UgaXMiLAogICAgIlRoZSBiZXN0IHRoaW5nIGFib3V0IHdvcmtpbmcgaW4gcHJvZHVjdCBtYW5hZ2Vt"
        "ZW50IGlzIiwKXQoKCmRlZiBsb2FkKG1vZGVsX25hbWUpOgogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gdHJhbnNmb3JtZXJz"
        "IGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplcgogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHN5cy5leGl0KAogICAg"
        "ICAgICAgICAiTWlzc2luZyBkZXBlbmRlbmNpZXMuIFJ1bjpcbiIKICAgICAgICAgICAgIiAgcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLnR4dFxuIgog"
        "ICAgICAgICAgICAiVGhpcyBsYWIgbmVlZHMgdG9yY2ggYW5kIHRyYW5zZm9ybWVycyAofjJHQikuIFNlZSAuLi9TRVRVUC5tZFxuIgogICAgICAgICAgICAi"
        "Zm9yIHRoZSBDb2xhYiByb3V0ZSBpZiB5b3UnZCByYXRoZXIgbm90IGluc3RhbGwgbG9jYWxseS4iCiAgICAgICAgKQogICAgcHJpbnQoZiJMb2FkaW5nIHtt"
        "b2RlbF9uYW1lfSAuLi4iKQogICAgcHJpbnQoIihGaXJzdCBydW4gZG93bmxvYWRzIHRoZSBtb2RlbC4gU2VlIC4uL0NVUlJFTlQubWQuKVxuIikKICAgIHRv"
        "ayA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJh"
        "aW5lZCgKICAgICAgICBtb2RlbF9uYW1lLCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDMyCiAgICApCiAgICBtb2RlbC5ldmFsKCkKICAgIHJldHVybiB0b3Jj"
        "aCwgdG9rLCBtb2RlbAoKCmRlZiBzaG93X3Rva2VuaXphdGlvbih0b2ssIHRleHQpOgogICAgIiIiQmVmb3JlIGFueXRoaW5nIGVsc2U6IHRoZSBtb2RlbCBk"
        "b2VzIG5vdCBzZWUgd29yZHMuIiIiCiAgICBpZHMgPSB0b2suZW5jb2RlKHRleHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSkKICAgIHBpZWNlcyA9IFt0"
        "b2suZGVjb2RlKFtpXSkgZm9yIGkgaW4gaWRzXQogICAgcHJpbnQoIj0iICogNzApCiAgICBwcmludCgiRklSU1QsIFRIRSBUT0tFTlMiKQogICAgcHJpbnQo"
        "Ij0iICogNzApCiAgICBwcmludChmJyAgSW5wdXQ6ICJ7dGV4dH0iJykKICAgIHByaW50KGYiICBCZWNvbWVzIHtsZW4oaWRzKX0gdG9rZW5zOiIpCiAgICBw"
        "cmludCgiICAgIiwgIiB8ICIuam9pbihyZXByKHApIGZvciBwIGluIHBpZWNlcykpCiAgICBwcmludCgpCiAgICBwcmludCgiICBOb3RlIHRoZSBsZWFkaW5n"
        "IHNwYWNlcyBhdHRhY2hlZCB0byB3b3Jkcy4gJyBQYXJpcycgYW5kICdQYXJpcyciKQogICAgcHJpbnQoIiAgYXJlIGRpZmZlcmVudCB0b2tlbnMgd2l0aCBk"
        "aWZmZXJlbnQgSURzLiIpCiAgICBwcmludCgpCgogICAgIyB0aGUgc3RyYXdiZXJyeSBkZW1vbnN0cmF0aW9uCiAgICBmb3Igd29yZCBpbiAoInN0cmF3YmVy"
        "cnkiLCAidW5iZWxpZXZhYmxlIik6CiAgICAgICAgd2lkID0gdG9rLmVuY29kZSh3b3JkLCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICAgICAgd3Ag"
        "PSBbdG9rLmRlY29kZShbaV0pIGZvciBpIGluIHdpZF0KICAgICAgICBwcmludChmJyAgInt3b3JkfSIgLT4ge2xlbih3aWQpfSB0b2tlbihzKTogeyIgfCAi"
        "LmpvaW4ocmVwcihwKSBmb3IgcCBpbiB3cCl9JykKICAgIHByaW50KCkKICAgIHByaW50KCIgIFRoZSBtb2RlbCBjYW5ub3Qgc2VlIGxldHRlcnMuIEFza2lu"
        "ZyBpdCB0byBjb3VudCB0aGUgcidzIGluIikKICAgIHByaW50KCIgICdzdHJhd2JlcnJ5JyBhc2tzIGl0IHRvIHJlYXNvbiBhYm91dCB0aGUgaW50ZXJuYWwg"
        "Y29tcG9zaXRpb24gb2YiKQogICAgcHJpbnQoIiAgc3ltYm9scyBpdCB3YXMgbmV2ZXIgc2hvd24uIEl0IGlzIGEgcmVwcmVzZW50YXRpb24gcHJvYmxlbSwg"
        "YW5kIikKICAgIHByaW50KCIgIG5vIGFtb3VudCBvZiBwcm9tcHRpbmcgZml4ZXMgYSByZXByZXNlbnRhdGlvbiBwcm9ibGVtLiIpCiAgICBwcmludCgpCgoK"
        "ZGVmIHN0ZXBfcmVwb3J0KHRvcmNoLCB0b2ssIG1vZGVsLCBwcm9tcHQsIHRlbXBlcmF0dXJlLCB0b3Bfbiwgc3RlcHMpOgogICAgIiIiR2VuZXJhdGUgdG9r"
        "ZW4gYnkgdG9rZW4sIHByaW50aW5nIHRoZSBkaXN0cmlidXRpb24gYXQgZWFjaCBzdGVwLiIiIgogICAgaWRzID0gdG9rLmVuY29kZShwcm9tcHQsIHJldHVy"
        "bl90ZW5zb3JzPSJwdCIpCiAgICBnZW5lcmF0ZWQgPSBbXQoKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoZiJHRU5FUkFUSU5HIEFUIFRFTVBFUkFU"
        "VVJFIHt0ZW1wZXJhdHVyZX0iKQogICAgcHJpbnQoIj0iICogNzApCiAgICBwcmludChmJyAgUHJvbXB0OiAie3Byb21wdH0iXG4nKQoKICAgIGZvciBzdGVw"
        "IGluIHJhbmdlKHN0ZXBzKToKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gbW9kZWwoaWRzKQogICAgICAgIGxvZ2l0"
        "cyA9IG91dC5sb2dpdHNbMCwgLTEsIDpdICAgICAgICAgICMgc2NvcmVzIGZvciBFVkVSWSB0b2tlbiwgdGhpcyBwb3NpdGlvbgoKICAgICAgICAjIHRoZSBz"
        "b2Z0bWF4IGZyb20gQ2hhcHRlciAxLCB3aXRoIHRlbXBlcmF0dXJlIGFwcGxpZWQKICAgICAgICBpZiB0ZW1wZXJhdHVyZSA8PSAwOgogICAgICAgICAgICBw"
        "cm9icyA9IHRvcmNoLnplcm9zX2xpa2UobG9naXRzKQogICAgICAgICAgICBwcm9ic1tpbnQodG9yY2guYXJnbWF4KGxvZ2l0cykuaXRlbSgpKV0gPSAxLjAK"
        "ICAgICAgICBlbHNlOgogICAgICAgICAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobG9naXRzIC8gdGVtcGVyYXR1cmUsIGRpbT0tMSkKCiAgICAgICAgdG9w"
        "ID0gdG9yY2gudG9wayhwcm9icywgdG9wX24pCiAgICAgICAgcHJpbnQoZiIgIFN0ZXAge3N0ZXAgKyAxfTogYWZ0ZXIgXCJ7cHJvbXB0fXsnJy5qb2luKGdl"
        "bmVyYXRlZCl9XCIiKQogICAgICAgIHByaW50KGYiICAgIHsndG9rZW4nOjwxNn17J3Byb2InOj45fSAgIHsncmF3IGxvZ2l0Jzo+MTB9IikKICAgICAgICBm"
        "b3IgcmFuayBpbiByYW5nZSh0b3Bfbik6CiAgICAgICAgICAgIHRpZCA9IHRvcC5pbmRpY2VzW3JhbmtdLml0ZW0oKQogICAgICAgICAgICBwID0gdG9wLnZh"
        "bHVlc1tyYW5rXS5pdGVtKCkKICAgICAgICAgICAgYmFyID0gIiMiICogbWF4KDEsIGludChwICogMzQpKQogICAgICAgICAgICBwcmludChmIiAgICB7cmVw"
        "cih0b2suZGVjb2RlKFt0aWRdKSlbOjE1XTo8MTZ9e3A6PjguMyV9ICAgIgogICAgICAgICAgICAgICAgICBmIntsb2dpdHNbdGlkXS5pdGVtKCk6PjEwLjJm"
        "fSAge2Jhcn0iKQoKICAgICAgICAjIHdoYXQgZnJhY3Rpb24gb2YgYWxsIHByb2JhYmlsaXR5IHNpdHMgaW4gdGhlIHRvcCBmZXc/CiAgICAgICAgaGVhZCA9"
        "IHRvcC52YWx1ZXMuc3VtKCkuaXRlbSgpCiAgICAgICAgcHJpbnQoZiIgICAgLT4gdG9wIHt0b3Bfbn0gaG9sZCB7aGVhZDouMSV9IG9mIGFsbCBwcm9iYWJp"
        "bGl0eTsgIgogICAgICAgICAgICAgIGYidGhlIG90aGVyIHtsZW4ocHJvYnMpIC0gdG9wX246LH0gdG9rZW5zIHNoYXJlIHsxIC0gaGVhZDouMSV9IikKCiAg"
        "ICAgICAgIyBzYW1wbGUKICAgICAgICBpZiB0ZW1wZXJhdHVyZSA8PSAwOgogICAgICAgICAgICBueHQgPSB0b3JjaC50b3BrKHByb2JzLCAxKS5pbmRpY2Vz"
        "CiAgICAgICAgZWxzZToKICAgICAgICAgICAgbnh0ID0gdG9yY2gubXVsdGlub21pYWwocHJvYnMsIG51bV9zYW1wbGVzPTEpCiAgICAgICAgcGllY2UgPSB0"
        "b2suZGVjb2RlKG54dCkKICAgICAgICBnZW5lcmF0ZWQuYXBwZW5kKHBpZWNlKQogICAgICAgIHByaW50KGYiICAgIC0+IHNhbXBsZWQ6IHtwaWVjZSFyfVxu"
        "IikKCiAgICAgICAgaWRzID0gdG9yY2guY2F0KFtpZHMsIG54dC51bnNxdWVlemUoMCldLCBkaW09MSkKCiAgICBwcmludChmJyAgUmVzdWx0OiAie3Byb21w"
        "dH17IiIuam9pbihnZW5lcmF0ZWQpfSInKQogICAgcHJpbnQoKQoKCmRlZiBmaW5kX25lYXJfdGllcyh0b3JjaCwgdG9rLCBtb2RlbCwgcHJvbXB0LCBzdGVw"
        "cz0yNSwgdGhyZXNob2xkPTAuMDMpOgogICAgIiIiCiAgICBGaW5kIHRoZSBicmFuY2ggcG9pbnRzOiBzdGVwcyB3aGVyZSB0aGUgdG9wIHR3byB0b2tlbnMg"
        "YXJlIGNsb3NlIGVub3VnaAogICAgdGhhdCBhIHJvdW5kaW5nIGRpZmZlcmVuY2UgY291bGQgZmxpcCB0aGVtLiBUaGVzZSBhcmUgZXhhY3RseSB3aGVyZQog"
        "ICAgJ2J1dCBpdCB3b3JrZWQgeWVzdGVyZGF5JyBjb21lcyBmcm9tLgogICAgIiIiCiAgICBpZHMgPSB0b2suZW5jb2RlKHByb21wdCwgcmV0dXJuX3RlbnNv"
        "cnM9InB0IikKICAgIHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoIk5FQVItVElFUzogV0hFUkUgUkVQUk9EVUNJQklMSVRZIEdPRVMgVE8gRElFIikKICAg"
        "IHByaW50KCI9IiAqIDcwKQogICAgcHJpbnQoZiIgIFdhbGtpbmcge3N0ZXBzfSBzdGVwcyBhdCB0ZW1wZXJhdHVyZSAwIChhbHdheXMgdGFrZSB0aGUgdG9w"
        "IHRva2VuKS4iKQogICAgcHJpbnQoZiIgIEZsYWdnaW5nIGFueSBzdGVwIHdoZXJlIHRoZSB0b3AgdHdvIGFyZSB3aXRoaW4ge3RocmVzaG9sZDouMCV9Llxu"
        "IikKCiAgICBmb3VuZCA9IDAKICAgIHRleHQgPSBwcm9tcHQKICAgIGZvciBzdGVwIGluIHJhbmdlKHN0ZXBzKToKICAgICAgICB3aXRoIHRvcmNoLm5vX2dy"
        "YWQoKToKICAgICAgICAgICAgbG9naXRzID0gbW9kZWwoaWRzKS5sb2dpdHNbMCwgLTEsIDpdCiAgICAgICAgcHJvYnMgPSB0b3JjaC5zb2Z0bWF4KGxvZ2l0"
        "cywgZGltPS0xKQogICAgICAgIHRvcCA9IHRvcmNoLnRvcGsocHJvYnMsIDIpCiAgICAgICAgZ2FwID0gKHRvcC52YWx1ZXNbMF0gLSB0b3AudmFsdWVzWzFd"
        "KS5pdGVtKCkKICAgICAgICBhLCBiID0gdG9rLmRlY29kZSh0b3AuaW5kaWNlc1swOjFdKSwgdG9rLmRlY29kZSh0b3AuaW5kaWNlc1sxOjJdKQoKICAgICAg"
        "ICBpZiBnYXAgPCB0aHJlc2hvbGQ6CiAgICAgICAgICAgIGZvdW5kICs9IDEKICAgICAgICAgICAgcHJpbnQoZiIgIFN0ZXAge3N0ZXAgKyAxOj4yfSAgZ2Fw"
        "IHtnYXA6LjRmfSAgICIKICAgICAgICAgICAgICAgICAgZiJ7YSFyfSAoe3RvcC52YWx1ZXNbMF06LjNmfSkgIHZzICB7YiFyfSAoe3RvcC52YWx1ZXNbMV06"
        "LjNmfSkiKQoKICAgICAgICBueHQgPSB0b3AuaW5kaWNlc1swOjFdCiAgICAgICAgdGV4dCArPSB0b2suZGVjb2RlKG54dCkKICAgICAgICBpZHMgPSB0b3Jj"
        "aC5jYXQoW2lkcywgbnh0LnVuc3F1ZWV6ZSgwKV0sIGRpbT0xKQoKICAgIHByaW50KCkKICAgIGlmIGZvdW5kOgogICAgICAgIHByaW50KGYiICBGb3VuZCB7"
        "Zm91bmR9IG5lYXItdGllKHMpIGluIHtzdGVwc30gc3RlcHMuIikKICAgICAgICBwcmludCgpCiAgICAgICAgcHJpbnQoIiAgQXQgZWFjaCBvZiB0aG9zZSwg"
        "dHdvIGRpZmZlcmVudCBvdXRwdXRzIHdlcmUgbmVhcmx5IGVxdWFsbHkiKQogICAgICAgIHByaW50KCIgIGxpa2VseS4gSW4gcHJvZHVjdGlvbiwgcmVxdWVz"
        "dHMgZ2V0IGJhdGNoZWQgb24gdGhlIEdQVSBhbmQiKQogICAgICAgIHByaW50KCIgIGZsb2F0aW5nLXBvaW50IGFkZGl0aW9uIGlzbid0IGFzc29jaWF0aXZl"
        "LCBzbyB0aGUgYXJpdGhtZXRpYyIpCiAgICAgICAgcHJpbnQoIiAgY2FuIGNvbWUgb3V0IGZyYWN0aW9uYWxseSBkaWZmZXJlbnRseSBydW4gdG8gcnVuLiBB"
        "dCBhIHN0ZXAiKQogICAgICAgIHByaW50KCIgIGxpa2UgdGhlc2UsIHRoYXQgZGlmZmVyZW5jZSBmbGlwcyB3aGljaCB0b2tlbiB3aW5zLiIpCiAgICAgICAg"
        "cHJpbnQoKQogICAgICAgIHByaW50KCIgIFRoaXMgaXMgd2h5IHRlbXBlcmF0dXJlIDAgaXMgbm90IGRldGVybWluaXN0aWMgaW4gcHJhY3RpY2UsIikKICAg"
        "ICAgICBwcmludCgiICBldmVuIHRob3VnaCBpdCBpcyBpbiB0aGVvcnkuIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoIiAgTm8gbmVhci10aWVzIHRoaXMg"
        "dGltZS4gVHJ5IGEgbW9yZSBvcGVuLWVuZGVkIHByb21wdCDigJQiKQogICAgICAgIHByaW50KCcgIGZhY3R1YWwgY29tcGxldGlvbnMgdGVuZCB0byBiZSBj"
        "b25maWRlbnQgYWxsIHRoZSB3YXkgdGhyb3VnaC4nKQogICAgcHJpbnQoKQoKCmRlZiBtYWluKCk6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIo"
        "ZGVzY3JpcHRpb249IkxhYiBCOiB3YXRjaCB0aGUgbW9kZWwgY2hvb3NlLiIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9IlF3ZW4v"
        "UXdlbjIuNS0wLjVCLUluc3RydWN0IiwKICAgICAgICAgICAgICAgICAgIGhlbHA9InNlZSAuLi9DVVJSRU5ULm1kIikKICAgIHAuYWRkX2FyZ3VtZW50KCIt"
        "LXByb21wdCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXRlbXBlcmF0dXJlIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjgpCiAgICBw"
        "LmFkZF9hcmd1bWVudCgiLS10b3AtbiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwKQogICAgcC5hZGRfYXJndW1lbnQoIi0tc3RlcHMiLCB0eXBlPWludCwgZGVm"
        "YXVsdD02KQogICAgcC5hZGRfYXJndW1lbnQoIi0tbmVhci10aWVzIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgIGhlbHA9Imh1"
        "bnQgZm9yIGJyYW5jaCBwb2ludHMgaW5zdGVhZCBvZiBnZW5lcmF0aW5nIikKICAgIGFyZ3MgPSBwLnBhcnNlX2FyZ3MoKQoKICAgIHRvcmNoLCB0b2ssIG1v"
        "ZGVsID0gbG9hZChhcmdzLm1vZGVsKQogICAgcHJvbXB0ID0gYXJncy5wcm9tcHQgb3IgUFJPTVBUU1swXQoKICAgIHNob3dfdG9rZW5pemF0aW9uKHRvaywg"
        "cHJvbXB0KQoKICAgIGlmIGFyZ3MubmVhcl90aWVzOgogICAgICAgIGZpbmRfbmVhcl90aWVzKHRvcmNoLCB0b2ssIG1vZGVsLCBhcmdzLnByb21wdCBvciBQ"
        "Uk9NUFRTWzFdKQogICAgICAgIHJldHVybgoKICAgIHN0ZXBfcmVwb3J0KHRvcmNoLCB0b2ssIG1vZGVsLCBwcm9tcHQsIGFyZ3MudGVtcGVyYXR1cmUsIGFy"
        "Z3MudG9wX24sIGFyZ3Muc3RlcHMpCgogICAgcHJpbnQoIj0iICogNzApCiAgICBwcmludCgiTk9XIFRSWSBUSEVTRSIpCiAgICBwcmludCgiPSIgKiA3MCkK"
        "ICAgIHByaW50KCIgIFNhbWUgcHJvbXB0LCB0aHJlZSB0ZW1wZXJhdHVyZXMg4oCUIHdhdGNoIHRoZSBkaXN0cmlidXRpb24gZmxhdHRlbjoiKQogICAgcHJp"
        "bnQoIiAgICBweXRob24gd2F0Y2hfaXRfY2hvb3NlLnB5IC0tdGVtcGVyYXR1cmUgMC4yIikKICAgIHByaW50KCIgICAgcHl0aG9uIHdhdGNoX2l0X2Nob29z"
        "ZS5weSAtLXRlbXBlcmF0dXJlIDAuOCIpCiAgICBwcmludCgiICAgIHB5dGhvbiB3YXRjaF9pdF9jaG9vc2UucHkgLS10ZW1wZXJhdHVyZSAxLjUiKQogICAg"
        "cHJpbnQoKQogICAgcHJpbnQoIiAgQSBjb25maWRlbnQgcHJvbXB0IHZzIGFuIG9wZW4tZW5kZWQgb25lOiIpCiAgICBwcmludCgnICAgIHB5dGhvbiB3YXRj"
        "aF9pdF9jaG9vc2UucHkgLS1wcm9tcHQgIlRoZSBjYXBpdGFsIG9mIEZyYW5jZSBpcyInKQogICAgcHJpbnQoJyAgICBweXRob24gd2F0Y2hfaXRfY2hvb3Nl"
        "LnB5IC0tcHJvbXB0ICJNeSBmYXZvdXJpdGUgdGhpbmcgYWJvdXQgVHVlc2RheSBpcyInKQogICAgcHJpbnQoKQogICAgcHJpbnQoIiAgVGhlIGJyYW5jaCBw"
        "b2ludHMgd2hlcmUgcmVwcm9kdWNpYmlsaXR5IGJyZWFrczoiKQogICAgcHJpbnQoIiAgICBweXRob24gd2F0Y2hfaXRfY2hvb3NlLnB5IC0tbmVhci10aWVz"
        "IikKICAgIHByaW50KCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=="
    ),
}

for name, blob in FILES.items():
    pathlib.Path(name).write_bytes(base64.b64decode(blob))
    print('wrote', name)


In [ ]:
!python watch_it_choose.py


## Tokens, then the distribution

In [ ]:
!python watch_it_choose.py --steps 5

### Look for

- **Where it's certain vs where it isn't.** After "The capital of France is"
  the distribution is a spike. Mid-sentence in open prose it's flat across
  dozens of options.
- **How little of the probability mass the top 10 sometimes hold.**

## Temperature is a flatness dial

Run all three. Watch the tail thicken.

In [ ]:
!python watch_it_choose.py --steps 3 --temperature 0.2
!python watch_it_choose.py --steps 3 --temperature 0.8
!python watch_it_choose.py --steps 3 --temperature 1.5

## The branch points

Steps where the top two tokens are within 3% of each other. At each one, two
different outputs were nearly equally likely.

In production, requests get batched on the GPU and floating-point addition
isn't associative, so the arithmetic comes out fractionally differently run to
run. At a near-tie, that flips which token wins.

**This is why temperature 0 isn't deterministic in practice.**

In [ ]:
!python watch_it_choose.py --near-ties